# Domain Generalization

This notebook implements **Objective 3 / Research Question 2** of the project.

The notebook demonstrates how domain-generalization techniques can improve
performance on the **RealWaste** dataset after training on **TrashNet**.

Techniques evaluated:

- Heavy Data Augmentation
- Mixup
- Heavy Augmentation + Mixup

Workflow

1. Load TrashNet pretrained model
2. Replace classifier head with a 9-class RealWaste classifier
3. Fine-tune using a small reserved RealWaste subset
4. Evaluate on the held-out RealWaste dataset
5. Save the best model and evaluation results

In [1]:
import json
import os

import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score

import config

from dataset import (
    get_clean_transforms,
    get_heavy_transforms,
    make_realwaste_splits,
    mixup_batch,
    set_seed,
)

from models import build_model

In [2]:
BACKBONE = "resnet50"

# Options:
# "resnet50"
# "efficientnet_b0"
# "mobilenet_v3_large"

TECHNIQUE = "both"

# Options:
# "heavy_aug"
# "mixup"
# "both"

EPOCHS = 10

MIXUP_ALPHA = 0.2

In [3]:
def replace_head_for_realwaste(
        model,
        backbone_name,
        num_classes):

    backbone_name = backbone_name.lower()

    if backbone_name == "resnet50":

        in_features = model.fc.in_features

        model.fc = nn.Linear(
            in_features,
            num_classes
        )

    elif backbone_name == "efficientnet_b0":

        in_features = model.classifier[1].in_features

        model.classifier[1] = nn.Linear(
            in_features,
            num_classes
        )

    elif backbone_name == "mobilenet_v3_large":

        in_features = model.classifier[3].in_features

        model.classifier[3] = nn.Linear(
            in_features,
            num_classes
        )

    return model

In [4]:
def evaluate(
    model,
    loader,
    device):

    model.eval()

    predictions = []
    labels_all = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)

            outputs = model(images)

            preds = outputs.argmax(dim=1)

            predictions.extend(
                preds.cpu().numpy()
            )

            labels_all.extend(
                labels.numpy()
            )

    accuracy = accuracy_score(
        labels_all,
        predictions
    )

    macro_f1 = f1_score(
        labels_all,
        predictions,
        average="macro"
    )

    return accuracy, macro_f1

In [5]:
def run_domain_generalization(
    backbone,
    technique,
    epochs=10,
    mixup_alpha=0.2):

    set_seed()

    device = config.DEVICE

    checkpoint = os.path.join(
        config.CHECKPOINT_DIR,
        f"{backbone}_trashnet_best.pt"
    )

    if not os.path.exists(checkpoint):

        raise FileNotFoundError(
            f"Checkpoint not found:\n{checkpoint}"
        )

    _, clean_eval_tf = get_clean_transforms()

    heavy_train_tf, heavy_eval_tf = get_heavy_transforms()

    use_heavy = technique in (
        "heavy_aug",
        "both"
    )

    finetune_tf = (
        heavy_train_tf
        if use_heavy
        else clean_eval_tf
    )

    (
        _,
        finetune_ds,
        eval_ds,
        realwaste_classes,
    ) = make_realwaste_splits(
        clean_eval_tf,
        finetune_tf
    )

    finetune_loader = DataLoader(
        finetune_ds,
        batch_size=config.BATCH_SIZE,
        shuffle=True,
        num_workers=2
    )

    eval_loader = DataLoader(
        eval_ds,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        num_workers=2
    )

    model, _ = build_model(
        backbone,
        len(config.TRASHNET_CLASSES),
        pretrained=False
    )

    model.load_state_dict(
        torch.load(
            checkpoint,
            map_location=device
        )
    )

    model = replace_head_for_realwaste(
        model,
        backbone,
        len(realwaste_classes)
    )

    model = model.to(device)

    return (
        model,
        finetune_loader,
        eval_loader,
        realwaste_classes
    )

In [6]:
model, train_loader, test_loader, classes = run_domain_generalization(
    BACKBONE,
    TECHNIQUE,
    EPOCHS,
    MIXUP_ALPHA
)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=config.WEIGHT_DECAY
)

best_f1 = -1

best_checkpoint = os.path.join(
    config.CHECKPOINT_DIR,
    f"{BACKBONE}_dg_{TECHNIQUE}_best.pt"
)

use_mixup = TECHNIQUE in (
    "mixup",
    "both"
)

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0

    for images, labels in train_loader:

        images = images.to(config.DEVICE)
        labels = labels.to(config.DEVICE)

        optimizer.zero_grad()

        if use_mixup:

            mixed_images, mixed_labels = mixup_batch(
                images,
                labels,
                len(classes),
                alpha=MIXUP_ALPHA
            )

            outputs = model(mixed_images)

            log_probs = torch.log_softmax(
                outputs,
                dim=1
            )

            loss = -(
                mixed_labels *
                log_probs
            ).sum(dim=1).mean()

        else:

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * images.size(0)

    train_loss = running_loss / len(train_loader.dataset)

    accuracy, macro_f1 = evaluate(
        model,
        test_loader,
        config.DEVICE
    )

    print(
        f"Epoch {epoch+1}/{EPOCHS}"
        f" | Loss={train_loss:.4f}"
        f" | Accuracy={accuracy:.4f}"
        f" | MacroF1={macro_f1:.4f}"
    )

    if macro_f1 > best_f1:

        best_f1 = macro_f1

        torch.save(
            model.state_dict(),
            best_checkpoint
        )

Epoch 1/10 | Loss=1.8256 | Accuracy=0.5295 | MacroF1=0.4185
Epoch 2/10 | Loss=1.3436 | Accuracy=0.6526 | MacroF1=0.6228
Epoch 3/10 | Loss=1.0050 | Accuracy=0.7204 | MacroF1=0.6975
Epoch 4/10 | Loss=1.0078 | Accuracy=0.7593 | MacroF1=0.7484
Epoch 5/10 | Loss=0.9626 | Accuracy=0.7557 | MacroF1=0.7510
Epoch 6/10 | Loss=0.7393 | Accuracy=0.7622 | MacroF1=0.7533
Epoch 7/10 | Loss=0.7670 | Accuracy=0.7917 | MacroF1=0.7879
Epoch 8/10 | Loss=0.7089 | Accuracy=0.7888 | MacroF1=0.7889
Epoch 9/10 | Loss=0.6102 | Accuracy=0.8248 | MacroF1=0.8231
Epoch 10/10 | Loss=0.6415 | Accuracy=0.8009 | MacroF1=0.8029


In [7]:
model.load_state_dict(
torch.load(
    best_checkpoint,
    map_location=config.DEVICE
)
)

final_accuracy, final_f1 = evaluate(
model,
test_loader,
config.DEVICE
)

print("\nFinal Results")

print("Accuracy :", final_accuracy)

print("Macro F1 :", final_f1)


Final Results
Accuracy : 0.8248290373487638
Macro F1 : 0.8230533734430696


In [8]:
results = {

    "backbone": BACKBONE,

    "technique": TECHNIQUE,

    "accuracy": final_accuracy,

    "macro_f1": final_f1,

    "checkpoint": best_checkpoint,

    "classes": classes
}

save_path = os.path.join(
    config.RESULTS_DIR,
    f"{BACKBONE}_dg_{TECHNIQUE}.json"
)

with open(save_path, "w") as f:
    json.dump(
        results,
        f,
        indent=4
    )

print("Results saved to")

print(save_path)

Results saved to
e:\wasteClassificationProject_FIXED\wasteClassificationProject\outputs\resnet50_dg_both.json
